# Coloriseur U-Net — entraînement (Colab)

Entraînement du modèle de colorisation (espace Lab, dataset STL-10) avec **Keras 3 + JAX**.

**Pipeline :** cloner le repo → backend JAX → dataset → créer *ou* recharger le modèle →
entraîner → courbes → récupérer le `.keras`.


## 1. Récupérer le code du projet

On clone le dépôt pour disposer des modules `ml/colorizer/`.

In [ ]:
import os

REPO_URL = "https://github.com/tristan-reig/Projet_L3_REMAKE.git"
REPO_DIR = "/content/projet-ia-creative"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Répertoire de travail :", os.getcwd())

## 2. Backend JAX + dépendances

`KERAS_BACKEND` doit être défini **avant** tout import de Keras. JAX et Keras 3 sont préinstallés sur Colab donc on installe les autres dépendances.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

# Versions épinglées pour éviter le conflit Protobuf sur Colab
!pip install -q "tensorflow-datasets==4.9.6" "protobuf>=5.28,<6" "tensorflow-metadata<1.17"

import keras
print("Keras :", keras.__version__, "| backend :", keras.backend.backend())

## 3. Paramètres d'entraînement

À ajuster selon le temps et le quota GPU disponibles.

In [ ]:
IMG_SIZE   = 128
BATCH_SIZE = 128
TRAIN_SIZE = 90000   # STL-10 : 100k images non labellisées ; le reste sert de validation
EPOCHS     = 20

MODEL_NAME = "model_colorizer.keras"

## 4. (Optionnel) Monter Google Drive

Recommandé : les checkpoints sont sauvegardés sur Drive pendant l'entraînement, donc une déconnexion de session ne fait pas tout perdre.

In [ ]:
USE_DRIVE = True  # passe à False pour rester en local Colab + download manuel

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/projet-ia-creative/models"
else:
    SAVE_DIR = "/content/projet-ia-creative/models"

os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
print("Le modèle sera sauvegardé ici :", MODEL_PATH)

## 5. Préparer le dataset

Pipeline STL-10 → Lab normalisé (entrée L, cible ab), avec augmentation.

In [ ]:
from ml.colorizer.data import build_dataset

train_ds, val_ds = build_dataset(batch_size=BATCH_SIZE, train_size=TRAIN_SIZE)
print("Datasets prêts.")

## 6. Créer un nouveau modèle, ou recharger l'existant

Pour faire un nouveau modèle, exécuter **6a**, pour affiner un modèle déjà entrainé , exécuter **6b**.

### 6a. Nouveau modèle (repartir de zéro)

In [ ]:
from ml.colorizer.model import build_unet, compile_model

model = build_unet(img_size=IMG_SIZE)
model = compile_model(model, train_size=TRAIN_SIZE, batch_size=BATCH_SIZE, epochs=EPOCHS)
model.summary()

### 6b. Recharger un modèle existant (pour continuer l'entraînement)

Si tu as déjà un `.keras` (sur Drive, ou uploadé via le panneau Fichiers de Colab), indique son chemin ici.

In [ ]:
# from ml.colorizer.model import colorization_loss, compile_model
# import keras
#
# RELOAD_PATH = MODEL_PATH  # ou "/content/mon_modele.keras" si upload manuel
# model = keras.models.load_model(RELOAD_PATH, custom_objects={"colorization_loss": colorization_loss})
# model = compile_model(model, train_size=TRAIN_SIZE, batch_size=BATCH_SIZE, epochs=EPOCHS)
# print("Modèle rechargé depuis", RELOAD_PATH)

## 7. Entraîner

`ModelCheckpoint` sauvegarde le meilleur modèle (selon `val_loss`) au fil des époques. `EarlyStopping` arrête si la validation ne progresse plus et restaure les meilleurs poids.

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping

callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 8. Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

h = history.history
pairs = [("loss", "val_loss", "Loss (colorization)")]
if "mae" in h:
    pairs.append(("mae", "val_mae", "MAE"))

fig, axes = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 4))
if len(pairs) == 1:
    axes = [axes]
for ax, (tr, val, title) in zip(axes, pairs):
    ax.plot(h[tr], label="Train", linewidth=2)
    ax.plot(h[val], label="Val", linewidth=2, linestyle="--")
    ax.set_title(title); ax.set_xlabel("Époque"); ax.legend()
    ax.grid(True, linestyle="--", alpha=0.6)
fig.tight_layout()
plt.show()

## 9. Tester la colorisation sur une image

Upload une image via le panneau Fichiers de Colab, puis indique son chemin.

In [ ]:
from PIL import Image
from ml.colorizer.colorize import colorize_image
import matplotlib.pyplot as plt

TEST_IMAGE = "/content/test.jpg"  # chemin à adapter

img = Image.open(TEST_IMAGE)
result = colorize_image(model, img)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img.convert("L"), cmap="gray"); axes[0].set_title("Entrée (N&B)"); axes[0].axis("off")
axes[1].imshow(result); axes[1].set_title("Colorisé"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 10. Récupérer le modèle entraîné

Si Drive est utilisé, le `.keras` y est déjà. Sinon, on peut le télécharger ici, puis le placer dans le dossier `models/` du projet (suivi par Git LFS).

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    files.download(MODEL_PATH)
else:
    print("Modèle disponible sur Drive :", MODEL_PATH)